In [1]:
# ============================================================
# 01_AURORA_download_datasets.ipynb
# AURORA-TWETF Dataset Downloader
#
# Purpose:
# 1. Download raw ETF, Taiwan market, cross-market, FX, and optional macro proxies.
# 2. Save individual raw yfinance files.
# 3. Save unified close-price, volume, return, and OHLCV panels.
# 4. Save data inventory for reproducibility.
#
# This notebook does NOT create features or labels.
# Feature engineering starts in:
# 02_AURORA_data_preparation_leakage_controlled.ipynb
# ============================================================

from __future__ import annotations

import os
import sys
import json
import time
import random
import hashlib
import subprocess
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Google Drive mount skipped or failed.")

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    try:
        return __import__(import_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return __import__(import_name)

yf = install_if_missing("yfinance", "yfinance")

import numpy as np
import pandas as pd

# ============================================================
# 1. Reproducibility and project paths
# ============================================================

RANDOM_SEED = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

PROJECT_CODE = "AURORA_TWETF"

PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw_yfinance"
PANEL_DATA_DIR = DATA_ROOT / "panels"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
DOCS_DIR = PUBLICATION_ROOT / "docs"

for d in [
    PUBLICATION_ROOT,
    DATA_ROOT,
    RAW_DATA_DIR,
    PANEL_DATA_DIR,
    MODELING_DIR,
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    DOCS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("AURORA-TWETF Dataset Downloader")
print("=" * 80)
print("Timestamp UTC:", RUN_TIMESTAMP)
print("Project root :", PUBLICATION_ROOT)
print("Raw data dir :", RAW_DATA_DIR)
print("Panel dir    :", PANEL_DATA_DIR)
print("Output root  :", OUTPUT_ROOT)
print("=" * 80)

# ============================================================
# 2. Dataset configuration
# ============================================================

START_DATE = "2021-01-01"
END_DATE = None

# Taiwan ETF allocation universe
ETF_TICKERS = {
    "0050": "0050.TW",
    "006208": "006208.TW",
    "00692": "00692.TW",
    "00881": "00881.TW",
}

# Main Taiwan market and cross-market variables
MARKET_TICKERS = {
    "TAIEX": "^TWII",
    "SP500": "^GSPC",
    "NASDAQ": "^IXIC",
    "SOXX": "SOXX",
    "USD_TWD": "TWD=X",
}

# Optional additional global risk proxies
# You can remove these if you want a smaller dataset.
OPTIONAL_TICKERS = {
    "VIX": "^VIX",
    "NIKKEI225": "^N225",
    "HANGSENG": "^HSI",
    "KOSPI": "^KS11",
    "US10Y": "^TNX",
}

# Combine all tickers
ALL_TICKERS = {}
ALL_TICKERS.update(ETF_TICKERS)
ALL_TICKERS.update(MARKET_TICKERS)
ALL_TICKERS.update(OPTIONAL_TICKERS)

# This list defines what will be required for the core experiment.
REQUIRED_SYMBOLS = list(ETF_TICKERS.keys()) + ["TAIEX"]

print("ETF universe:")
for k, v in ETF_TICKERS.items():
    print(f"  {k}: {v}")

print("\nMarket / cross-market tickers:")
for k, v in MARKET_TICKERS.items():
    print(f"  {k}: {v}")

print("\nOptional tickers:")
for k, v in OPTIONAL_TICKERS.items():
    print(f"  {k}: {v}")

# ============================================================
# 3. Utility functions
# ============================================================

def safe_filename(symbol_name, ticker):
    ticker_clean = (
        str(ticker)
        .replace("^", "")
        .replace(".", "_")
        .replace("=", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
    )
    return f"{symbol_name}_{ticker_clean}.csv"

def flatten_yfinance_columns(df):
    """
    yfinance sometimes returns MultiIndex columns.
    This function flattens them into simple strings.
    """
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join([str(x) for x in col if str(x) != ""])
            for col in df.columns
        ]
    return df

def find_column(df, target_field):
    """
    Find a column containing target_field.
    Example: target_field='Close' finds 'Close' or 'Close_0050.TW'.
    """
    target = target_field.lower().replace(" ", "")
    candidates = []

    for col in df.columns:
        c = str(col).lower().replace(" ", "")
        if target in c:
            candidates.append(col)

    if candidates:
        # Prefer exact match if available
        for c in candidates:
            if str(c).lower().replace(" ", "") == target:
                return c
        return candidates[0]

    return None

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

# ============================================================
# 4. Download one ticker with cache
# ============================================================

def download_one_ticker(symbol_name, ticker, start=START_DATE, end=END_DATE, force_download=False):
    """
    Download one ticker from yfinance and cache it as CSV.
    If the cached file exists, it is loaded instead of downloaded.
    """
    cache_path = RAW_DATA_DIR / safe_filename(symbol_name, ticker)

    if cache_path.exists() and not force_download:
        try:
            df = pd.read_csv(cache_path, parse_dates=["Date"])
            df = df.set_index("Date").sort_index()
            return df, "loaded_from_cache", cache_path
        except Exception as exc:
            print(f"Cache load failed for {symbol_name}, re-downloading. Error: {exc}")

    print(f"Downloading {symbol_name}: {ticker}")

    try:
        data = yf.download(
            ticker,
            start=start,
            end=end,
            auto_adjust=False,
            progress=False,
            threads=False,
        )
    except Exception as exc:
        print(f"ERROR downloading {symbol_name} / {ticker}: {exc}")
        return pd.DataFrame(), "download_error", cache_path

    if data is None or data.empty:
        print(f"WARNING: no data returned for {symbol_name} / {ticker}")
        return pd.DataFrame(), "empty_download", cache_path

    data = flatten_yfinance_columns(data)
    data = data.reset_index()

    # Standardize date column
    if "Date" not in data.columns:
        first_col = data.columns[0]
        data = data.rename(columns={first_col: "Date"})

    data["Date"] = pd.to_datetime(data["Date"])
    data = data.sort_values("Date")

    data.to_csv(cache_path, index=False, encoding="utf-8-sig")

    df = data.set_index("Date").sort_index()

    return df, "downloaded", cache_path

# ============================================================
# 5. Download all datasets
# ============================================================

raw_data = {}
inventory_rows = []

for symbol_name, ticker in ALL_TICKERS.items():
    df_raw, status, cache_path = download_one_ticker(
        symbol_name=symbol_name,
        ticker=ticker,
        start=START_DATE,
        end=END_DATE,
        force_download=False,
    )

    if df_raw.empty:
        inventory_rows.append({
            "symbol_name": symbol_name,
            "ticker": ticker,
            "status": status,
            "cache_path": str(cache_path),
            "n_rows": 0,
            "start_date": None,
            "end_date": None,
            "open_column": None,
            "high_column": None,
            "low_column": None,
            "close_column": None,
            "adj_close_column": None,
            "volume_column": None,
            "is_required": symbol_name in REQUIRED_SYMBOLS,
        })
        continue

    raw_data[symbol_name] = df_raw

    open_col = find_column(df_raw, "Open")
    high_col = find_column(df_raw, "High")
    low_col = find_column(df_raw, "Low")
    close_col = find_column(df_raw, "Close")
    adj_close_col = find_column(df_raw, "Adj Close")
    volume_col = find_column(df_raw, "Volume")

    # Prefer adjusted close if available, otherwise close.
    preferred_close = adj_close_col if adj_close_col is not None else close_col

    inventory_rows.append({
        "symbol_name": symbol_name,
        "ticker": ticker,
        "status": status,
        "cache_path": str(cache_path),
        "n_rows": int(df_raw.shape[0]),
        "start_date": df_raw.index.min().strftime("%Y-%m-%d"),
        "end_date": df_raw.index.max().strftime("%Y-%m-%d"),
        "open_column": open_col,
        "high_column": high_col,
        "low_column": low_col,
        "close_column": close_col,
        "adj_close_column": adj_close_col,
        "preferred_close_column": preferred_close,
        "volume_column": volume_col,
        "is_required": symbol_name in REQUIRED_SYMBOLS,
    })

    time.sleep(0.2)

data_inventory = pd.DataFrame(inventory_rows)

inventory_path = TABLE_DIR / "table_01_raw_data_inventory.csv"
data_inventory.to_csv(inventory_path, index=False, encoding="utf-8-sig")

print("\nRaw data inventory:")
print(data_inventory.to_string(index=False))

# ============================================================
# 6. Validate required symbols
# ============================================================

missing_required = []

for symbol in REQUIRED_SYMBOLS:
    if symbol not in raw_data or raw_data[symbol].empty:
        missing_required.append(symbol)

if missing_required:
    raise RuntimeError(
        "Missing required data symbols: "
        + ", ".join(missing_required)
        + "\nCannot continue. Check yfinance access or ticker names."
    )

print("\nAll required symbols downloaded or loaded successfully.")

# ============================================================
# 7. Build unified OHLCV panels
# ============================================================

close_series = {}
open_series = {}
high_series = {}
low_series = {}
volume_series = {}

for _, row in data_inventory.iterrows():
    symbol = row["symbol_name"]

    if symbol not in raw_data:
        continue

    df_raw = raw_data[symbol]

    open_col = row.get("open_column", None)
    high_col = row.get("high_column", None)
    low_col = row.get("low_column", None)
    preferred_close_col = row.get("preferred_close_column", None)
    volume_col = row.get("volume_column", None)

    if isinstance(preferred_close_col, str) and preferred_close_col in df_raw.columns:
        close_series[symbol] = df_raw[preferred_close_col].astype(float).rename(symbol)

    if isinstance(open_col, str) and open_col in df_raw.columns:
        open_series[symbol] = df_raw[open_col].astype(float).rename(symbol)

    if isinstance(high_col, str) and high_col in df_raw.columns:
        high_series[symbol] = df_raw[high_col].astype(float).rename(symbol)

    if isinstance(low_col, str) and low_col in df_raw.columns:
        low_series[symbol] = df_raw[low_col].astype(float).rename(symbol)

    if isinstance(volume_col, str) and volume_col in df_raw.columns:
        volume_series[symbol] = df_raw[volume_col].astype(float).rename(symbol)

close_panel = pd.concat(close_series.values(), axis=1).sort_index()
open_panel = pd.concat(open_series.values(), axis=1).sort_index() if open_series else pd.DataFrame()
high_panel = pd.concat(high_series.values(), axis=1).sort_index() if high_series else pd.DataFrame()
low_panel = pd.concat(low_series.values(), axis=1).sort_index() if low_series else pd.DataFrame()
volume_panel = pd.concat(volume_series.values(), axis=1).sort_index() if volume_series else pd.DataFrame()

# Daily returns from preferred close
return_panel = close_panel.pct_change().replace([np.inf, -np.inf], np.nan)

# ETF-only panels
etf_close_panel = close_panel[[c for c in ETF_TICKERS.keys() if c in close_panel.columns]].copy()
etf_return_panel = etf_close_panel.pct_change().replace([np.inf, -np.inf], np.nan)

# ============================================================
# 8. Save panels
# ============================================================

close_panel_path = PANEL_DATA_DIR / "AURORA_close_panel.parquet"
open_panel_path = PANEL_DATA_DIR / "AURORA_open_panel.parquet"
high_panel_path = PANEL_DATA_DIR / "AURORA_high_panel.parquet"
low_panel_path = PANEL_DATA_DIR / "AURORA_low_panel.parquet"
volume_panel_path = PANEL_DATA_DIR / "AURORA_volume_panel.parquet"
return_panel_path = PANEL_DATA_DIR / "AURORA_return_panel.parquet"
etf_close_panel_path = PANEL_DATA_DIR / "AURORA_etf_close_panel.parquet"
etf_return_panel_path = PANEL_DATA_DIR / "AURORA_etf_return_panel.parquet"

close_panel.to_parquet(close_panel_path)
open_panel.to_parquet(open_panel_path)
high_panel.to_parquet(high_panel_path)
low_panel.to_parquet(low_panel_path)
volume_panel.to_parquet(volume_panel_path)
return_panel.to_parquet(return_panel_path)
etf_close_panel.to_parquet(etf_close_panel_path)
etf_return_panel.to_parquet(etf_return_panel_path)

# Also save CSV versions for inspection
close_panel.to_csv(PANEL_DATA_DIR / "AURORA_close_panel.csv", encoding="utf-8-sig")
volume_panel.to_csv(PANEL_DATA_DIR / "AURORA_volume_panel.csv", encoding="utf-8-sig")
return_panel.to_csv(PANEL_DATA_DIR / "AURORA_return_panel.csv", encoding="utf-8-sig")
etf_close_panel.to_csv(PANEL_DATA_DIR / "AURORA_etf_close_panel.csv", encoding="utf-8-sig")
etf_return_panel.to_csv(PANEL_DATA_DIR / "AURORA_etf_return_panel.csv", encoding="utf-8-sig")

print("\nSaved panel files:")
print(close_panel_path)
print(volume_panel_path)
print(return_panel_path)
print(etf_close_panel_path)
print(etf_return_panel_path)

# ============================================================
# 9. Create missing-value and date-coverage reports
# ============================================================

coverage_rows = []

for col in close_panel.columns:
    s = close_panel[col].dropna()

    coverage_rows.append({
        "symbol_name": col,
        "n_total_rows": int(close_panel.shape[0]),
        "n_nonmissing_close": int(s.shape[0]),
        "missing_close_count": int(close_panel[col].isna().sum()),
        "missing_close_rate": float(close_panel[col].isna().mean()),
        "first_valid_date": s.index.min().strftime("%Y-%m-%d") if len(s) else None,
        "last_valid_date": s.index.max().strftime("%Y-%m-%d") if len(s) else None,
    })

coverage_report = pd.DataFrame(coverage_rows)

coverage_path = TABLE_DIR / "table_02_data_coverage_report.csv"
coverage_report.to_csv(coverage_path, index=False, encoding="utf-8-sig")

missing_by_date = close_panel.isna().sum(axis=1).rename("n_missing_symbols").reset_index()
missing_by_date = missing_by_date.rename(columns={"Date": "date", "index": "date"})

missing_by_date_path = TABLE_DIR / "table_03_missing_symbols_by_date.csv"
missing_by_date.to_csv(missing_by_date_path, index=False, encoding="utf-8-sig")

print("\nCoverage report:")
print(coverage_report.to_string(index=False))

# ============================================================
# 10. Save dataset download configuration
# ============================================================

download_config = {
    "project_code": PROJECT_CODE,
    "timestamp_utc": RUN_TIMESTAMP,
    "random_seed": RANDOM_SEED,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "publication_root": str(PUBLICATION_ROOT),
    "raw_data_dir": str(RAW_DATA_DIR),
    "panel_data_dir": str(PANEL_DATA_DIR),
    "etf_tickers": ETF_TICKERS,
    "market_tickers": MARKET_TICKERS,
    "optional_tickers": OPTIONAL_TICKERS,
    "required_symbols": REQUIRED_SYMBOLS,
    "output_files": {
        "close_panel": str(close_panel_path),
        "open_panel": str(open_panel_path),
        "high_panel": str(high_panel_path),
        "low_panel": str(low_panel_path),
        "volume_panel": str(volume_panel_path),
        "return_panel": str(return_panel_path),
        "etf_close_panel": str(etf_close_panel_path),
        "etf_return_panel": str(etf_return_panel_path),
        "data_inventory": str(inventory_path),
        "coverage_report": str(coverage_path),
        "missing_by_date": str(missing_by_date_path),
    },
}

config_path = REPORT_DIR / "AURORA_dataset_download_config.json"
save_json(config_path, download_config)

# ============================================================
# 11. Save file manifest
# ============================================================

manifest = make_file_manifest(DATA_ROOT)
manifest_path = DOCS_DIR / "AURORA_dataset_file_manifest_SHA256.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")

# ============================================================
# 12. Console summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF DATASET DOWNLOAD COMPLETE")
print("=" * 80)
print("Timestamp UTC:", RUN_TIMESTAMP)
print("Raw data directory :", RAW_DATA_DIR)
print("Panel data directory:", PANEL_DATA_DIR)
print("Inventory table    :", inventory_path)
print("Coverage report    :", coverage_path)
print("Download config    :", config_path)
print("File manifest      :", manifest_path)

print("\nClose panel shape:", close_panel.shape)
print("Close panel date range:", close_panel.index.min().date(), "to", close_panel.index.max().date())

print("\nETF close panel shape:", etf_close_panel.shape)
print("ETF close panel columns:", list(etf_close_panel.columns))

print("\nReturn panel shape:", return_panel.shape)

print("\nNext notebook:")
print("02_AURORA_data_preparation_leakage_controlled.ipynb")
print("=" * 80)

Mounted at /content/drive
AURORA-TWETF Dataset Downloader
Timestamp UTC: 2026-06-23T13:35:41Z
Project root : /content/drive/MyDrive/AURORA_TWETF
Raw data dir : /content/drive/MyDrive/AURORA_TWETF/data/raw_yfinance
Panel dir    : /content/drive/MyDrive/AURORA_TWETF/data/panels
Output root  : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF
ETF universe:
  0050: 0050.TW
  006208: 006208.TW
  00692: 00692.TW
  00881: 00881.TW

Market / cross-market tickers:
  TAIEX: ^TWII
  SP500: ^GSPC
  NASDAQ: ^IXIC
  SOXX: SOXX
  USD_TWD: TWD=X

Optional tickers:
  VIX: ^VIX
  NIKKEI225: ^N225
  HANGSENG: ^HSI
  KOSPI: ^KS11
  US10Y: ^TNX

Raw data inventory:
symbol_name    ticker     status                                                                 cache_path  n_rows start_date   end_date    open_column    high_column    low_column        close_column    adj_close_column preferred_close_column    volume_column  is_required
       0050   0050.TW downloaded     /content/drive/MyDrive/AUROR